[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/01_mobility.ipynb)

# 01 — UE Mobility Generation

**Purpose.** Generate UE trajectories over the road network with SUMO and map
them into the Sionna-RT scene frame. PROJECT.md section 16, Phase 2 and
section 9.1.

A trajectory is

```text
T_u = {(t_k, x_u,k, y_u,k, v_u,k, d_u,k)}
```

— time, position, speed and heading per UE. These positions are where notebook
02 evaluates RSRP, so this notebook decides *where the network is measured*, and
therefore what the UE-weighted Band Priority Score is weighted by.

**Inputs.** The road network from the scenario, plus the UE population and
mobility parameters for this scenario.

**Outputs.** UE trajectories in the scene coordinate frame, and the UE spatial
distribution notebook 02 turns into a density grid.

**Requires** `uv sync --extra sumo` (or a system SUMO with `SUMO_HOME` set).

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook except the COLAB_PACKAGES line below, which names
# the extras this particular notebook needs. Forked the repository? Change these
# three values and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [
    ("hydra", "hydra-core"),
    ("traci", "eclipse-sumo"),
    ("sumolib", "eclipse-sumo"),
    ("pyproj", "pyproj"),  # not a transitive eclipse-sumo dependency; src.mobility.frame needs it
]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/external/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Why the mobility model is not a detail

The five KPIs are computed over a spatial grid, but two of them are weighted by
where UEs actually are. Band Priority Score is a UE-weighted average
(PROJECT.md section 4.7): it asks whether *users* are on the right frequency
layer, not whether *area* is. A mobility model that puts every UE on one arterial
road produces a Band Priority Score about that road.

So the UE distribution is an input to the objective, not scenery. Section 12.2
perturbs it deliberately — count, density, routes, speed, arrival times — and
notebook 06 has to report whether the optimized configuration survived that.

## 3. Road network

SUMO moves UEs over the road network; Sionna-RT propagates through the 3D
geometry. Both describe the same place and must agree about where it is.

In [ ]:
from src.mobility import frame, network

net = network.load_net(cfg)
summary = network.network_summary(net)
print(f"edges: {summary['edge_count']:,}")
print(f"total lane length: {summary['total_lane_length_m']:,.0f} m")
print(f"network bbox (SUMO local frame): {summary['bbox']}")

# The frame transform is cheap (reads two small XML headers, not the network
# or the meshes) so it is worth deriving here too, before any demand or
# simulation cost is spent.
scene_frame = frame.derive_frame(cfg)
xmin, ymin, xmax, ymax = summary["bbox"]
net_bbox_scene = (
    xmin + scene_frame.dx, ymin + scene_frame.dy,
    xmax + scene_frame.dx, ymax + scene_frame.dy,
)
print(f"network bbox (scene frame): {net_bbox_scene}")
print(f"scene bounds:               {scene_frame.bounds}")

sx0, sy0, sx1, sy1 = scene_frame.bounds
if net_bbox_scene[0] < sx0 or net_bbox_scene[1] < sy0 or net_bbox_scene[2] > sx1 or net_bbox_scene[3] > sy1:
    print(
        "\nThe road network extends beyond the scene on at least one side — "
        "trajectories WILL be clipped, not merely bounded. See section 6 and "
        "configs/mobility.yaml:clip. This is expected, not an error."
    )

## 4. UE population and demand

PROJECT.md section 9.1 lists what a scenario may vary: UE count, density,
routes, trajectories, speed and arrival times. Each is a scenario parameter, so
each belongs in the manifest from notebook 00 — not hardcoded here.

In [ ]:
from pathlib import Path

from src.data import scenario as scenario_module
from src.mobility import demand

fingerprint = scenario_module.scene_fingerprint(cfg)
scenario_id = scenario_module.scenario_id(cfg, fingerprint)
sumo_dir = Path(cfg.mobility.output.dir) / f"scenario_id={scenario_id}" / "sumo"

trips = demand.build_trips(cfg, sumo_dir)
routes = demand.build_routes(cfg, trips, sumo_dir / "routes.rou.xml")
print(f"scenario_id: {scenario_id}")
print(f"trips:  {trips}")
print(f"routes: {routes}")

## 5. Run SUMO and collect trajectories

One record per UE per sampling step. The sampling interval sets how many RSRP
evaluations notebook 02 has to perform, so it is a cost decision as much as a
fidelity one.

In [ ]:
from src.mobility import simulate

fcd_path = simulate.run_sumo(cfg, routes, sumo_dir)
raw = simulate.read_fcd(fcd_path, cfg)
print(f"raw FCD samples: {len(raw):,}  (collector: {cfg.mobility.run.collector})")
raw.head()

## 6. Map SUMO coordinates into the scene frame

PROJECT.md section 9.1: SUMO coordinates are mapped into the Sionna-RT scene
frame. This is the single most likely place for a silent error in this notebook
— an offset or a flipped axis produces trajectories that look fine in isolation
and sit in the wrong streets.

Check it against the buildings, not against itself.

In [ ]:
tidied = simulate.tidy(raw, cfg, scenario_id, scene_frame)
trajectories, clip_stats = frame.clip_to_scene(tidied, scene_frame, cfg)
print(
    f"clipped_fraction: {clip_stats['clipped_fraction']:.1%}  "
    f"(rows_in={clip_stats['rows_in']}, rows_out={clip_stats['rows_out']}, "
    f"ues_dropped={clip_stats['ues_dropped']})"
)

# A quick look against the scene extent. Overlaying the building footprints
# from scene.poly.xml is left to src.evaluation.analysis once it is
# implemented; this is the position check this section needs first.
fig, ax = plt.subplots(figsize=(7, 7))
xmin, ymin, xmax, ymax = scene_frame.bounds
ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, lw=1.5))
ax.scatter(trajectories["sim_x"], trajectories["sim_y"], s=2, alpha=0.3)
ax.set(xlabel="sim_x (m)", ylabel="sim_y (m)", title="Clipped UE trajectories, scene frame")
ax.set_aspect("equal")

## 7. Sanity-check the trajectories

Speeds within the road limits, no teleports between consecutive samples, no UE
inside a building, and an arrival profile that matches what was configured.

In [ ]:
from src.mobility import checks

check_results = checks.run_all(trajectories, net, cfg)
for name, violations in check_results.items():
    print(f"{name}: {len(violations)} violation(s)")
check_results["speed_within_limits"].head()

## 8. UE spatial distribution

The output that matters downstream. Notebook 02 grids this into the UE density
`rho` used by the Band Priority Score, on **the same grid geometry** as the
radio map — `src.data.ue_density.build_grid` owns that, and nothing else may
define it.

In [ ]:
from src.radio import scene as scene_module

xmin, ymin, xmax, ymax = scene_module.scene_bounds(cfg)
fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, lw=1.5))
ax.scatter(trajectories["sim_x"], trajectories["sim_y"], s=2, alpha=0.3)
ax.set(xlabel="sim_x (m)", ylabel="sim_y (m)", title="UE positions over the scene extent")
ax.set_aspect("equal")

# TODO: grid onto cfg.radio.grid resolution and report the empty-cell fraction
# once src.data.ue_density.build_grid / ue_density are implemented — both are
# still stubs, so this section stops at the scatter plot for now.

## 9. Persist with provenance

Trajectories are a scenario artifact. They carry `scenario_id` so that notebook
03 can split on it, and the mobility parameters and seed so the scenario can be
regenerated exactly (PROJECT.md section 22.4).

In [ ]:
from src.mobility import network, persist

manifest = scenario_module.build_manifest(
    cfg,
    scenario_id,
    sumo={"version": network.sumo_version(network.sumo_binary("sumo"))},
    seeds=demand.seeds(cfg),
    frame={"dx": scene_frame.dx, "dy": scene_frame.dy, "scene_bounds": scene_frame.bounds},
    clipping=clip_stats,
    checks={name: len(v) for name, v in check_results.items()},
)
out_dir = persist.save_trajectories(trajectories, manifest, cfg)
print(f"wrote {out_dir}")

## 10. Handoff checklist

- [ ] The clipped fraction was reported and looked at — a nonzero value is
      expected (the network extends beyond the scene); an unexpectedly large
      one means the network and the scene do not describe the same place.
- [ ] Trajectories were eyeballed against the scene extent, not just asserted
      to be inside it.
- [ ] Speeds and arrival times match the configured mobility parameters —
      section 7's check results were looked at, not just counted.
- [ ] `scenario_id` is attached to every record.
- [ ] The seed and every mobility parameter are stored with the output.
- [ ] The UE distribution was looked at. If it is concentrated on a few roads,
      say so now — the Band Priority Score will inherit that concentration and
      notebook 06 will have to interpret it.